In [ ]:
"""
🟡 B-2. 응용 — 층을 쌓고, hidden 을 뜯어본다  (보통 · 목표 50분)

06번에서 강사와 함께 본 것을 **직접 손으로 확인**한다. 여기서 진짜로 익힐 것은
"2층이 더 좋다/나쁘다"가 아니라 **`hidden` 안에 무엇이 어떻게 들어 있는가** 다.

    hidden 의 모양 = (층수, 배치, 은닉차원)
                      ↑     ↑     ↑
                   몇 층?  몇 문장?  요약 벡터 크기

빈칸(TODO)은 5개다. 정답은 solutions/ 에 있다.
"""

In [ ]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

try:
    _HERE = Path(__file__).resolve().parent
except NameError:
    _HERE = Path.cwd()
sys.path.insert(0, str(_HERE.parent))

from imdb_data import load_imdb
from textutils import tokenize_en, build_vocab, pad_and_tensor
from model import LstmClassifier

torch.manual_seed(0)

## 1) 손으로 확인 — hidden 안에는 무엇이 들어 있나

가짜 입력으로 먼저 감을 잡는다. 훈련 없이 shape만 본다.

In [ ]:
x = torch.randn(8, 30, 16)          # 배치 8개 · 길이 30 · 입력차원 16

# TODO 1: 은닉차원 24 짜리 **2층 LSTM** 을 만든다. batch_first=True 를 잊지 말자.
lstm2 = None

if lstm2 is None:
    raise SystemExit("→ TODO 1 을 먼저 채우자. nn.LSTM(입력차원, 은닉차원, num_layers=?, batch_first=True)")

lstm2.eval()
with torch.no_grad():
    output, (h_n, c_n) = lstm2(x)   # LSTM 이라 짝으로 받는다

print("  output :", tuple(output.shape))
print("  h_n    :", tuple(h_n.shape))
print("  c_n    :", tuple(c_n.shape))

# TODO 2: 아래 빈칸을 **먼저 예상해서 적고** 나서 실행해 맞는지 확인하자.
#   output 의 모양은 (____, ____, ____) 이고, 가운데 30은 무엇을 뜻하는가?
#   h_n 의 첫 번째 숫자 2는 무엇을 뜻하는가?
#   c_n 이 h_n 과 모양이 같은 이유는?

## 2) 어느 것이 어느 층인가 — 직접 대조한다

06번에서 확인한 것: `hidden[-1]` 은 **맨 위 층의 마지막 은닉**이고,
그것은 `output[:, -1, :]` 와 같다. 정말인지 여기서 다시 재 보자.

In [ ]:
# TODO 3: torch.equal 로 두 가지를 비교해 True/False 를 출력한다.
#   (a) h_n[-1]  vs  output[:, -1, :]
#   (b) h_n[0]   vs  output[:, -1, :]
print("  h_n[-1] == output[:, -1, :] :", None)
print("  h_n[0]  == output[:, -1, :] :", None)

print("""
  하나는 True, 하나는 False 여야 한다. 어느 쪽이 True 인가?
    False 로 나온 쪽은 **몇 번째 층**을 가리키고 있는 걸까?
""")

## 3) dropout 이 정말 층 사이에만 걸리는가

같은 입력을 두 번 넣었을 때 결과가 달라지면 dropout 이 **켜져 있다**는 뜻이다.

In [ ]:
lstm_dp = nn.LSTM(16, 24, num_layers=2, batch_first=True, dropout=0.5)

# TODO 4: train 모드와 eval 모드에서 각각 두 번씩 통과시켜, 결과가 같은지 확인한다.
#   힌트: model.train() / model.eval() · torch.allclose(a, b)
#   예상: eval 은 항상 같고, train 은 매번 다르다.

## 4) 실제로 돌려 비교 — 1층 vs 2층

여기서 05번의 원칙을 지켜야 한다.

> 차이(신호)가 **재실행 흔들림(잡음)** 보다 뚜렷할 때만 "낫다"고 말한다.

In [ ]:
VOCAB_SIZE, MAX_LEN = 2000, 100
BATCH, LR, EPOCHS = 64, 1e-3, 8

train, val = load_imdb(n_train=5000, n_val=2000)
ttok = [tokenize_en(t) for t in train["text"]]
vtok = [tokenize_en(t) for t in val["text"]]
word2idx, _ = build_vocab(ttok, max_size=VOCAB_SIZE)
X_train = pad_and_tensor(ttok, word2idx, MAX_LEN)
y_train = torch.tensor(train["label"], dtype=torch.float32)
X_val = pad_and_tensor(vtok, word2idx, MAX_LEN)
y_val = torch.tensor(val["label"], dtype=torch.float32)


def run(num_layers, dropout, seed):
    torch.manual_seed(seed)
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH, shuffle=True)
    m = LstmClassifier(len(word2idx), 32, 32,
                       num_layers=num_layers, dropout=dropout)
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    crit = nn.BCELoss()
    best = 0.0
    for _ in range(EPOCHS):
        m.train()
        for xb, yb in loader:
            opt.zero_grad(); crit(m(xb), yb).backward(); opt.step()
        m.eval()
        with torch.no_grad():
            best = max(best, ((m(X_val) > 0.5).float() == y_val).float().mean().item())
    return best


# TODO 5: 아래 설정들을 **각각 시드 2개**로 돌려 표를 채운다.
#   설정: (1층, dropout 0.0) · (2층, dropout 0.3) · (2층, dropout 0.7)
#   시드: 42, 43
#   힌트: for 문 두 겹이면 된다. run(층수, dropout, 시드) 를 부르면 최고 정확도가 나온다.

results = {}     # {"이름": [시드42 결과, 시드43 결과]}

## 5) 판단 — 무엇을 말해도 되나

In [ ]:
if results:
    print(f"\n  {'설정':<22} {'결과':<20} {'평균':>8} {'흔들림':>8}")
    print("  " + "-" * 60)
    for name, scores in results.items():
        spread = max(scores) - min(scores)
        print(f"  {name:<22} {str([f'{s:.3f}' for s in scores]):<20} "
              f"{sum(scores)/len(scores):>8.3f} {spread:>8.3f}")

    noise = max(max(v) - min(v) for v in results.values())
    print(f"\n  잡음(같은 설정 재실행 최대 흔들림) = {noise:.3f}")
    print("  → 설정 사이의 평균 차이가 이 값의 2배를 넘는 것만 '차이가 있다'고 말하자.")
else:
    print("  (TODO 5 를 채우면 표가 나온다)")

## 기록하고 이야기하자

| 설정 | 시드 42 | 시드 43 | 평균 | 흔들림 |
|---|---|---|---|---|
| LSTM 1층 dropout 0.0 | | | | |
| LSTM 2층 dropout 0.3 | | | | |
| LSTM 2층 dropout 0.7 | | | | |

**내가 내린 결론**:

> 이 조건에서 층을 쌓는 것은 ______________________ .
> 근거: 신호 ______ vs 잡음 ______ .

**회고 때 나눌 것**
1. dropout 0.7 은 어땠나? 너무 세게 끄면 무슨 일이 일어나는가?
2. 2층이 확실히 이기게 하려면 무엇을 바꿔야 할까? (힌트: 데이터 크기·`MAX_LEN`)
3. `h_n[0]` 을 잘못 쓰면 어떤 일이 벌어지는지 한 문장으로 설명해 보자.